In the cells below is 'old' code that require some small additions to be included in the study material

### ...) Undertow velocity under wave through

In breaking waves, the mass transport towards the coast between wave crest and wave
trough may be quite large, resulting in rather large seaward-directed velocities under
the wave trough level

In [ ]:
# Page 202
def W3_u0(x_range, H0, T, d0, slope, angle):

    # The environmental conditions
    x = x_range               # the horizontal axis
    zbed = -(d0 - slope * x)  # bed elevation [m]
    h = -zbed                 # still water depth [m]
    h[h < 0] = 0              # no negative depths

    # given conditions
    gamma = 0.8               # wave breaking ratio
    rho = 1025                # Density of water [kg/m3]
    
    # The wave characteristics at every location in the cross-section
    L = np.array([wave_length(T, h) for h in h])  # The wavelength
    c = L/T                                       # The wave celerity
    k = 2*np.pi/L                                 # The wave number
    n = 0.5 + (k*h/np.sinh(2*k*h))                
    cg = n*c                                      # The wave group celerity
    Ksh = np.sqrt(cg[0]/cg)                       # The shoaling parameter
    
    '''Completed the code here'''
    
    H = ...                                 # The wave height due to shoaling and refraction
    Hbreaking = gamma * h                         # The wave-breaking height
    H[H>Hbreaking]=Hbreaking[H>Hbreaking]         # Adjusting the wave height
    g = 9.81
    E = 1/8*rho*g*H**2                            # The wave energy

    ''' finish the code here'''
    
    alpha = 1
    Er = ...
    
    q_non_break = ... = E/c
    q_roller = ... = alpha*Er/C
    q_drift = q_non_break + q_roller

    q_drift = q_drift[H == Hbreaking]

    u_trough = q_drift*np.cos(np.deg2rad(angle))/(rho*h)
    
    
    
    return u_trough

### ) The influence of wave setup

The setup caused by waves influences the water depth. This might influence the waves as they propagate, especially in shallow water, were the setup is . The code below calculates the water depth iterative, if you complete it with the code you made before.

In [ ]:
def W3_wave_setup2(x_range, H0, T, d0, slope, angle):

    fig, axs = plt.subplots(nrows=2,ncols=1,figsize=(5,4), sharex=True, sharey = False)
    
           
    z_bed = -(d0 - slope * x_range)
    x0_id = np.argwhere(z_bed > 0)[0][0]  # first location where water depth = 0
    axs[1].plot(x_range, z_bed, color = 'k', label = 'bed')
    axs[1].plot(np.zeros(x0_id), color = 'grey', label = 'still water level' )
    axs[1].legend();

    # The environmental conditions
    x = x_range.copy()        # the horizontal axis
    zbed = -(d0 - slope * x)  # bed elevation [m]
    h = -zbed                 # still water depth [m]
    h[h < 0] = 0              # no negative depths
    setup = np.zeros(len(h))  # initial wave-induced setup

    # given conditions
    gamma = 0.8               # wave breaking ratio

    axs[0].plot(setup[h>z_bed], label = 'initial', color = 'grey')
    for i in range(3): # number of iterations considered
     
        # The wave characteristics at every location in the cross-section
        L = np.array([wave_length(T, h) for h in h])  # The wavelength
        c = L/T                                       # The wave celerity
        k = 2*np.pi/L                                 # The wave number
        n = 0.5 + (k*h/np.sinh(2*k*h))                
        cg = n*c                                      # The wave group celerity
        Ksh = np.sqrt(cg[0]/cg)                       # The shoaling parameter
        
        snell_constant = np.sin(np.deg2rad(angle))/c[0]  # apply snell's law
        theta_radians = np.arcsin(snell_constant * c)
        Kr = (np.cos(theta_radians[0])/np.cos(theta_radians))**0.5
        
        H = H0*Ksh*Kr                                  # The wave height due to shoaling and refraction
        Hbreaking = gamma * h                         # The wave-breaking height
        H[H>Hbreaking]=Hbreaking[H>Hbreaking]         # Adjusting the wave height
        g = 9.81
        E = 1/8*rho*g*H**2                            # The wave energy
    
        Sxx = (n-0.5+n*np.cos(theta_radians)**2)*E    # Radiant stresses

        setup = np.zeros(Sxx.shape) # here we create a vector for the setup
        for j in range(len(setup)-1):# key here is that setup[0] = 0 
            setup[j+1] = setup[j] - (Sxx[j+1]-Sxx[j])/(1000*g*h[j])

        setup_shoreline = setup[~np.isnan(setup)][-1] # get setup at the shoreline, which is the last non-Nan value
        setup[np.isnan(setup)] = setup_shoreline # apply the setup to the water level

        h = -zbed + setup         # the still water depth + setup
        h[np.isnan(h)] = 0
        h[h < 0] = 0

        x0_id = np.argwhere(z_bed > setup_shoreline)[0][0]  # first location where water depth = 0
        axs[0].plot(setup[0:x0_id], label = 'iteration ' + str(i))

    axs[0].legend()
    axs[1].legend()

    
    axs[0].set_xlim(520,x0_id + 50);